# External 7 Dataset Acquisition and Reconstruction
### Christopher Rizo — MIDS Capstone

## Purpose

This notebook reconstructs the **seven external-source evaluation datasets / eleven selected assays**
used in the team's AssayLoop comparison from the original public sources wherever possible.

The goal is to reproduce the same target screens used by the team before running new LLM + HSA / DepMap experiments.

### Target assays

1. Schukken — paired aneuploid-vs-near-euploid whole-genome screen
2. Abt–Wang — PARCB dependency
3. Wilson — MERCS high
4. Nguyen — K562 AraC sensitization/resistance target used by the team
5. Shan — HepG2 lovastatin
6. Shan — HeLa lovastatin
7. Shan — HeLa LDLR knockout
8. Herranz-Ors — colorectal organoid
9. Herranz-Ors — oesophageal organoid
10. Herranz-Ors — ovarian organoid
11. Samelson — T22-lowering CRISPRi

## Reproducibility rule

I do **not** silently replace a source or hit definition when the exact target cannot be recovered.
Every reconstructed screen is checked against the source gene/hit counts recorded in the team repository.

The known ambiguity is the Schukken paired screen:
- paper-stated rule = 188 hits
- top 194 by the authors' AneuploidRank = 194 hits
- the team's frozen screen has 194 hits, but the exact frozen hit identities are not published in the team repo

Both public reconstructions are retained and the ambiguity is explicitly reported.

In [ ]:
# 1) Setup
import os, sys, json, subprocess, shutil, hashlib, zipfile, re, pickle
from pathlib import Path

ROOT = Path("/content")
JEFF_REPO = ROOT / "capstone-jeff"
ASSAYLOOP = JEFF_REPO / "AssayLoop"
SRC = ROOT / "external7_sources"
BUILT = ROOT / "external7_built"

SRC.mkdir(exist_ok=True)
BUILT.mkdir(exist_ok=True)

if not JEFF_REPO.exists():
    subprocess.run(
        ["git","clone","--depth","1","--branch","Jeff",
         "https://github.com/jhouser220/capstone.git", str(JEFF_REPO)],
        check=True
    )

print("Repo:", JEFF_REPO)
print("Source dir:", SRC)
print("Build dir:", BUILT)

Repo: /content/capstone-jeff
Source dir: /content/external7_sources
Build dir: /content/external7_built


In [ ]:
# 2) Install the AssayLoop runtime
import shutil, os, subprocess

if shutil.which("uv") is None:
    subprocess.run(["bash","-lc","curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

subprocess.run(["uv","python","install","3.11"], check=True)
subprocess.run(["uv","sync","--extra","torch","--python","3.11"], cwd=ASSAYLOOP, check=True)

PY = ASSAYLOOP / ".venv/bin/python"
print("Python:", PY)

Python: /content/capstone-jeff/AssayLoop/.venv/bin/python


In [ ]:
# 3) Download helper
import requests, time

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (academic reproducibility; contact via repository metadata)"
})

def download(url, dest, *, expected_min_bytes=100):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size >= expected_min_bytes:
        print("exists:", dest.name, dest.stat().st_size)
        return dest

    print("download:", dest.name)
    with session.get(url, stream=True, timeout=180, allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    if dest.stat().st_size < expected_min_bytes:
        raise RuntimeError(f"{dest} is unexpectedly small: {dest.stat().st_size}")

    print("  bytes:", dest.stat().st_size)
    return dest

## Direct public sources

These URLs follow the exact source files documented in Jeff's `scripts/external/SOURCES.md`.

In [ ]:
# 4) Schukken — SD3 CRISPR beta scores
SCHUKKEN = download(
    "https://raw.githubusercontent.com/kschukken/Aneuploid_CRISPR_Screens/main/"
    "SupData/SD3-%20CRISPR%20Screen%20Beta%20Scores.xlsx",
    SRC / "schukken_SD3_beta_scores.xlsx",
    expected_min_bytes=1_000_000,
)

download: schukken_SD3_beta_scores.xlsx
  bytes: 10819529


In [ ]:
# 5) Abt–Wang — final filtered MAGeCK gene summary
ABT = download(
    "https://raw.githubusercontent.com/jfreeland01/Abt_Wang_E2F3_CRISPR/main/"
    "Zhang_count/merged_spacer_counts_mageck_test.gene_summary_FILT.txt",
    SRC / "abt_gene_summary_FILT.txt",
    expected_min_bytes=100_000,
)

download: abt_gene_summary_FILT.txt
  bytes: 1332013


In [ ]:
# 6) Wilson — Supplementary Table 2
WILSON = download(
    "https://raw.githubusercontent.com/M1gus/MitoER-CRISPR-Screen/main/"
    "Suplementary%20Tables/Supplementary%20Table%202_Whole%20Genome%20Pooled%20Screening%20Results.csv",
    SRC / "wilson_ST2_screen_results.csv",
    expected_min_bytes=100_000,
)

download: wilson_ST2_screen_results.csv
  bytes: 1910744


In [ ]:
# 7) Nguyen — Supplemental Tables 0–5
# PMC moved automated article-file access to its public AWS Open Data bucket.
# Find the current article version, locate mmc2.xlsx, and download it anonymously.

from pathlib import Path
import requests
import xml.etree.ElementTree as ET
from urllib.parse import quote

NGUYEN = SRC / "nguyen_mmc2_supplemental_tables.xlsx"

# Remove the bad HTML file from the earlier attempt.
if NGUYEN.exists():
    NGUYEN.unlink()

bucket = "https://pmc-oa-opendata.s3.amazonaws.com"
pmcid = "PMC11914169"

# 1. Find available article version(s)
r = requests.get(
    bucket,
    params={
        "list-type": "2",
        "prefix": f"{pmcid}.",
        "delimiter": "/",
    },
    timeout=60,
)
r.raise_for_status()

root = ET.fromstring(r.content)
ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

prefixes = [
    x.text.rstrip("/")
    for x in root.findall(".//s3:CommonPrefixes/s3:Prefix", ns)
]

print("PMC versions:", prefixes)

if not prefixes:
    raise RuntimeError(f"No AWS PMC version found for {pmcid}")

# Normally there is one final published version.
article_prefix = prefixes[-1] + "/"

# 2. List files for that version
r = requests.get(
    bucket,
    params={
        "list-type": "2",
        "prefix": article_prefix,
    },
    timeout=60,
)
r.raise_for_status()

root = ET.fromstring(r.content)

keys = [
    x.text
    for x in root.findall(".//s3:Contents/s3:Key", ns)
]

xlsx_keys = [
    k for k in keys
    if k.lower().endswith(".xlsx")
]

print("\nXLSX files:")
for k in xlsx_keys:
    print(" ", k)

# 3. Locate the supplemental table workbook
matches = [
    k for k in xlsx_keys
    if "mmc2" in k.lower()
]

if len(matches) != 1:
    raise RuntimeError(
        f"Expected one mmc2.xlsx file, found {len(matches)}: {matches}"
    )

key = matches[0]

# 4. Download anonymously from the public bucket
url = f"{bucket}/{quote(key, safe='/')}"

print("\nDownloading:", key)

with requests.get(url, stream=True, timeout=180) as r:
    r.raise_for_status()

    with open(NGUYEN, "wb") as f:
        for chunk in r.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)

print("bytes:", NGUYEN.stat().st_size)

# 5. Validate that this is an actual XLSX/ZIP file, not HTML
with open(NGUYEN, "rb") as f:
    magic = f.read(4)

assert magic == b"PK\x03\x04", (
    f"Downloaded file is not a valid XLSX. Header={magic!r}"
)

assert NGUYEN.stat().st_size > 1_000_000, (
    f"Workbook is unexpectedly small: {NGUYEN.stat().st_size}"
)

print("Nguyen workbook validated:", NGUYEN)

PMC versions: ['PMC11914169.1']

XLSX files:
  PMC11914169.1/BLOODA_ADV-2024-014157-mmc2.xlsx

Downloading: PMC11914169.1/BLOODA_ADV-2024-014157-mmc2.xlsx
bytes: 1898700
Nguyen workbook validated: /content/external7_sources/nguyen_mmc2_supplemental_tables.xlsx


In [ ]:
# 8) Shan — Additional file 2, Table S1
SHAN = download(
    "https://static-content.springer.com/esm/art%3A10.1186%2Fs13059-025-03531-8/"
    "MediaObjects/13059_2025_3531_MOESM2_ESM.xlsx",
    SRC / "shan_additional_file2_TableS1.xlsx",
    expected_min_bytes=100_000,
)

download: shan_additional_file2_TableS1.xlsx
  bytes: 3632595


## Samelson

The final-paper supplementary workbooks were originally supplied directly to the team.
The same primary/secondary-screen tables are publicly available in the bioRxiv/PMC preprint record:

- Supplemental Table 2 = all screen results
- Supplemental Table 3 = called hits

The builder below verifies the resulting gene/hit counts against the team's expected target before use.

In [ ]:
# 9) Samelson — public preprint supplementary tables
# Supplement 2 = all primary/secondary screen results
# Supplement 3 = called hit genes

from pathlib import Path
import requests
import xml.etree.ElementTree as ET
from urllib.parse import quote

TAU_ALL = SRC / "tau_mmc2_all_genes.xlsx"
TAU_HITS = SRC / "tau_mmc3_hits.xlsx"

# Remove failed tiny files from the earlier attempt.
for p in [TAU_ALL, TAU_HITS]:
    if p.exists():
        p.unlink()

bucket = "https://pmc-oa-opendata.s3.amazonaws.com"
pmcid = "PMC10312804"

# Find available article versions
r = requests.get(
    bucket,
    params={
        "list-type": "2",
        "prefix": f"{pmcid}.",
        "delimiter": "/",
    },
    timeout=60,
)
r.raise_for_status()

root = ET.fromstring(r.content)
ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

prefixes = [
    x.text.rstrip("/")
    for x in root.findall(".//s3:CommonPrefixes/s3:Prefix", ns)
]

print("PMC versions:", prefixes)

if not prefixes:
    raise RuntimeError(f"No AWS PMC version found for {pmcid}")

# Use the latest available version.
article_prefix = prefixes[-1] + "/"

# List files in that version
r = requests.get(
    bucket,
    params={
        "list-type": "2",
        "prefix": article_prefix,
    },
    timeout=60,
)
r.raise_for_status()

root = ET.fromstring(r.content)

keys = [
    x.text
    for x in root.findall(".//s3:Contents/s3:Key", ns)
]

xlsx_keys = [k for k in keys if k.lower().endswith(".xlsx")]

print("\nXLSX files:")
for k in xlsx_keys:
    print(" ", k)

def find_file(filename):
    matches = [
        k for k in xlsx_keys
        if k.lower().endswith("/" + filename.lower())
        or Path(k).name.lower() == filename.lower()
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename}, found {len(matches)}: {matches}"
        )
    return matches[0]

all_key = find_file("media-2.xlsx")
hits_key = find_file("media-3.xlsx")

def download_s3_xlsx(key, dest, min_bytes):
    url = f"{bucket}/{quote(key, safe='/')}"

    print("\nDownloading:", key)

    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()

        with open(dest, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    print("bytes:", dest.stat().st_size)

    # XLSX files are ZIP containers and should start with PK.
    with open(dest, "rb") as f:
        magic = f.read(4)

    assert magic == b"PK\x03\x04", (
        f"{dest.name} is not a valid XLSX. Header={magic!r}"
    )

    assert dest.stat().st_size > min_bytes, (
        f"{dest.name} is unexpectedly small: {dest.stat().st_size}"
    )

download_s3_xlsx(all_key, TAU_ALL, 1_000_000)
download_s3_xlsx(hits_key, TAU_HITS, 100_000)

print("\nSamelson files validated:")
print("All genes:", TAU_ALL)
print("Hits:", TAU_HITS)

PMC versions: ['PMC10312804.2', 'PMC10312804.3', 'PMC10312804.4']

XLSX files:
  PMC10312804.4/media-1.xlsx
  PMC10312804.4/media-2.xlsx
  PMC10312804.4/media-3.xlsx
  PMC10312804.4/media-4.xlsx
  PMC10312804.4/media-5.xlsx

Downloading: PMC10312804.4/media-2.xlsx
bytes: 2406413

Downloading: PMC10312804.4/media-3.xlsx
bytes: 197261

Samelson files validated:
All genes: /content/external7_sources/tau_mmc2_all_genes.xlsx
Hits: /content/external7_sources/tau_mmc3_hits.xlsx


## Herranz-Ors organoid data

The organoid target requires:

- Nature Supplementary Table 5: binary BAGEL2 `is_depleted` calls
- Figshare `data_covariates_ALL-organoids.csv`

The cell below discovers the download URLs from the current Nature article and Figshare API rather than hard-coding a temporary signed URL.

In [ ]:
# 10) Herranz-Ors — locate Supplementary Table 5 by its actual schema
#
# Required columns:
# sample_ID, gene, ensembl_id, HGNC_ID, library, is_depleted

from pathlib import Path
import requests
import xml.etree.ElementTree as ET
from urllib.parse import quote
import zipfile
import pandas as pd
import re
import shutil

ORG_DIR = SRC / "organoid"
ORG_DIR.mkdir(exist_ok=True)

ORG_TABLE5 = ORG_DIR / "organoid_suppl_table5.tsv"
ORG_COV = SRC / "organoid_data_covariates_ALL-organoids.csv"

bucket = "https://pmc-oa-opendata.s3.amazonaws.com"
pmcid = "PMC13581617"

required = {
    "sample_ID",
    "gene",
    "ensembl_id",
    "HGNC_ID",
    "library",
    "is_depleted",
}

# ------------------------------------------------------------
# A) Get the PMC package listing
# ------------------------------------------------------------

r = requests.get(
    bucket,
    params={
        "list-type": "2",
        "prefix": f"{pmcid}.",
        "delimiter": "/",
    },
    timeout=60,
)
r.raise_for_status()

root = ET.fromstring(r.content)
ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

prefixes = [
    x.text.rstrip("/")
    for x in root.findall(".//s3:CommonPrefixes/s3:Prefix", ns)
]

print("PMC versions:", prefixes)

if not prefixes:
    raise RuntimeError(f"No PMC package found for {pmcid}")

article_prefix = prefixes[-1] + "/"

r = requests.get(
    bucket,
    params={
        "list-type": "2",
        "prefix": article_prefix,
    },
    timeout=60,
)
r.raise_for_status()

root = ET.fromstring(r.content)

objects = []

for item in root.findall(".//s3:Contents", ns):
    key = item.find("s3:Key", ns).text
    size = int(item.find("s3:Size", ns).text)
    objects.append((key, size))

print("\nArticle package files:")
for key, size in objects:
    print(f"{size:>12,}  {Path(key).name}")


# ------------------------------------------------------------
# B) Helpers
# ------------------------------------------------------------

def s3_download(key, dest):
    dest = Path(dest)

    url = f"{bucket}/{quote(key, safe='/')}"

    if dest.exists():
        dest.unlink()

    print("\nDownloading:", Path(key).name)

    with requests.get(url, stream=True, timeout=600) as rr:
        rr.raise_for_status()

        with open(dest, "wb") as f:
            for chunk in rr.iter_content(4 * 1024 * 1024):
                if chunk:
                    f.write(chunk)

    print("  bytes:", f"{dest.stat().st_size:,}")

    return dest


def check_dataframe(df, label):
    cols = {str(c).strip() for c in df.columns}

    if required.issubset(cols):
        print("\nFOUND TABLE 5:", label)
        print("Columns:", list(df.columns))
        print("Rows:", len(df))
        return True

    return False


def inspect_file(path):
    """
    Return a DataFrame if this file appears to be Supplementary Table 5.
    """
    path = Path(path)
    name = path.name.lower()

    try:
        if name.endswith((".xlsx", ".xls")):

            xls = pd.ExcelFile(path)

            for sheet in xls.sheet_names:
                try:
                    preview = pd.read_excel(
                        path,
                        sheet_name=sheet,
                        nrows=5
                    )

                    if required.issubset(
                        {str(c).strip() for c in preview.columns}
                    ):
                        full = pd.read_excel(
                            path,
                            sheet_name=sheet
                        )

                        if check_dataframe(
                            full,
                            f"{path.name} / sheet={sheet}"
                        ):
                            return full
                except Exception:
                    pass

        elif name.endswith(".csv"):

            preview = pd.read_csv(path, nrows=5)

            if required.issubset(
                {str(c).strip() for c in preview.columns}
            ):
                full = pd.read_csv(path)

                if check_dataframe(full, path.name):
                    return full

        elif name.endswith((".tsv", ".txt")):

            # Try tab first.
            for sep in ["\t", r"\s+"]:
                try:
                    preview = pd.read_csv(
                        path,
                        sep=sep,
                        engine="python",
                        nrows=5
                    )

                    if required.issubset(
                        {str(c).strip() for c in preview.columns}
                    ):
                        full = pd.read_csv(
                            path,
                            sep=sep,
                            engine="python"
                        )

                        if check_dataframe(full, path.name):
                            return full
                except Exception:
                    pass

    except Exception as e:
        print("Could not inspect", path.name, ":", e)

    return None


# ------------------------------------------------------------
# C) Search data files directly included in the PMC package
# ------------------------------------------------------------

table5_df = None

data_extensions = (
    ".zip",
    ".xlsx",
    ".xls",
    ".csv",
    ".tsv",
    ".txt",
)

candidate_objects = [
    (key, size)
    for key, size in objects
    if Path(key).name.lower().endswith(data_extensions)
]

print("\nCandidate data objects:", len(candidate_objects))


# ------------------------------------------------------------
# D) Download each candidate and inspect
# ------------------------------------------------------------

for key, size in candidate_objects:

    name = Path(key).name
    local = ORG_DIR / name

    try:
        s3_download(key, local)
    except Exception as e:
        print("Download failed:", name, e)
        continue

    # --------------------------------------------------------
    # ZIP: inspect members
    # --------------------------------------------------------

    if name.lower().endswith(".zip"):

        print("\nZIP contents:")

        with zipfile.ZipFile(local) as z:

            members = [
                n for n in z.namelist()
                if not n.endswith("/")
            ]

            for n in members:
                print(" ", n)

            # Prioritize anything that looks like Table 5.
            prioritized = sorted(
                members,
                key=lambda n: (
                    0 if re.search(
                        r"(table[\s_\-]*5|supp.*5)",
                        n,
                        flags=re.I
                    )
                    else 1,
                    n
                )
            )

            extract_dir = ORG_DIR / (local.stem + "_unzipped")
            extract_dir.mkdir(exist_ok=True)

            for member in prioritized:

                lower = member.lower()

                if not lower.endswith(
                    (".csv", ".tsv", ".txt", ".xlsx", ".xls")
                ):
                    continue

                target = extract_dir / Path(member).name

                with z.open(member) as src_f, open(target, "wb") as dst_f:
                    shutil.copyfileobj(src_f, dst_f)

                found = inspect_file(target)

                if found is not None:
                    table5_df = found
                    break

        if table5_df is not None:
            break

    # --------------------------------------------------------
    # Direct spreadsheet / text file
    # --------------------------------------------------------

    else:

        found = inspect_file(local)

        if found is not None:
            table5_df = found
            break


# ------------------------------------------------------------
# E) Validate and normalize Table 5
# ------------------------------------------------------------

if table5_df is None:
    raise RuntimeError(
        "Could not automatically identify Supplementary Table 5 "
        "inside the PMC article package. The package file listing "
        "printed above will show us the next exact target."
    )

missing = required - {
    str(c).strip()
    for c in table5_df.columns
}

assert not missing, f"Missing Table 5 columns: {missing}"

print("\nTable 5 rows:", f"{len(table5_df):,}")

# Jeff's builder reads whitespace/tab-separated text.
table5_df[
    [
        "sample_ID",
        "gene",
        "ensembl_id",
        "HGNC_ID",
        "library",
        "is_depleted",
    ]
].to_csv(
    ORG_TABLE5,
    sep="\t",
    index=False
)

print("Normalized Table 5:", ORG_TABLE5)
print("Normalized bytes:", f"{ORG_TABLE5.stat().st_size:,}")


# ------------------------------------------------------------
# F) Validate the existing Figshare covariates
# ------------------------------------------------------------

print("\nChecking existing covariate file:")

if not ORG_COV.exists():
    raise RuntimeError(
        "Covariate file is missing. We will reacquire it from Figshare."
    )

try:
    cov = pd.read_csv(ORG_COV)
except Exception as e:
    raise RuntimeError(
        f"Existing covariate file could not be read: {e}"
    )

print("Covariate bytes:", f"{ORG_COV.stat().st_size:,}")
print("Covariate rows:", len(cov))
print("Covariate columns:", list(cov.columns))

assert {"sample_ID", "tissue"}.issubset(cov.columns), (
    "The existing Figshare file does not contain sample_ID and tissue."
)

print("\n✅ Herranz-Ors assets validated")
print("Table 5   :", ORG_TABLE5)
print("Covariates:", ORG_COV)

PMC versions: ['PMC13581617.1']

Article package files:
     383,790  41586_2026_10830_Fig10_ESM.webp
     203,368  41586_2026_10830_Fig11_ESM.webp
     159,386  41586_2026_10830_Fig12_ESM.webp
     128,708  41586_2026_10830_Fig13_ESM.webp
     166,864  41586_2026_10830_Fig14_ESM.webp
     127,472  41586_2026_10830_Fig15_ESM.webp
      87,620  41586_2026_10830_Fig1_HTML.webp
      74,486  41586_2026_10830_Fig2_HTML.webp
      96,086  41586_2026_10830_Fig3_HTML.webp
     161,308  41586_2026_10830_Fig4_HTML.webp
     163,258  41586_2026_10830_Fig5_HTML.webp
      82,504  41586_2026_10830_Fig6_ESM.webp
     130,696  41586_2026_10830_Fig7_ESM.webp
     236,358  41586_2026_10830_Fig8_ESM.webp
      90,632  41586_2026_10830_Fig9_ESM.webp
     679,527  41586_2026_10830_MOESM1_ESM.pdf
   4,494,692  41586_2026_10830_MOESM2_ESM.pdf
 127,226,341  41586_2026_10830_MOESM3_ESM.zip
   3,334,031  41586_2026_10830_MOESM4_ESM.pdf
       2,756  PMC13581617.1.json
  23,205,283  PMC13581617.1.pdf
     133,

RuntimeError: Could not automatically identify Supplementary Table 5 inside the PMC article package. The package file listing printed above will show us the next exact target.

In [ ]:
# Fix Herranz-Ors Table 5 extraction from the already-downloaded supplement ZIP

from pathlib import Path
import zipfile
import pandas as pd

ORG_DIR = SRC / "organoid"
ORG_TABLE5 = ORG_DIR / "organoid_suppl_table5.tsv"
ORG_COV = SRC / "organoid_data_covariates_ALL-organoids.csv"

zip_path = ORG_DIR / "41586_2026_10830_MOESM3_ESM.zip"

assert zip_path.exists(), f"Missing ZIP: {zip_path}"

member = "supplementary_table_5_revision.csv"
raw_table5 = ORG_DIR / member

# Extract only Table 5
with zipfile.ZipFile(zip_path) as z:
    assert member in z.namelist(), f"{member} not found in ZIP"
    with z.open(member) as src_f, open(raw_table5, "wb") as dst_f:
        dst_f.write(src_f.read())

print("Extracted:", raw_table5)
print("Bytes:", raw_table5.stat().st_size)

# Important: despite the .csv extension, this file is whitespace-delimited.
table5_df = pd.read_csv(
    raw_table5,
    sep=r"\s+",
    engine="c"
)

print("\nColumns:")
print(list(table5_df.columns))

print("\nRows:", len(table5_df))
print("Samples:", table5_df["sample_ID"].nunique())

required = {
    "sample_ID",
    "gene",
    "ensembl_id",
    "HGNC_ID",
    "library",
    "is_depleted",
}

missing = required - set(table5_df.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

# Save normalized version for Jeff's builder
table5_df[
    [
        "sample_ID",
        "gene",
        "ensembl_id",
        "HGNC_ID",
        "library",
        "is_depleted",
    ]
].to_csv(
    ORG_TABLE5,
    sep="\t",
    index=False
)

print("\nNormalized Table 5:", ORG_TABLE5)
print("Normalized bytes:", ORG_TABLE5.stat().st_size)

# Validate the Figshare covariates you already downloaded.
cov = pd.read_csv(ORG_COV)

print("\nCovariate bytes:", ORG_COV.stat().st_size)
print("Covariate rows:", len(cov))
print("Covariate columns:", list(cov.columns))

assert {"sample_ID", "tissue"}.issubset(cov.columns), (
    "Covariate file is missing sample_ID and/or tissue."
)

print("\n✅ Herranz-Ors assets validated")

Extracted: /content/external7_sources/organoid/supplementary_table_5_revision.csv
Bytes: 171694934

Columns:
['sample_ID', 'gene', 'ensembl_id', 'HGNC_ID', 'library', 'is_depleted']

Rows: 2948036
Samples: 162

Normalized Table 5: /content/external7_sources/organoid/organoid_suppl_table5.tsv
Normalized bytes: 171694934

Covariate bytes: 7770
Covariate rows: 256
Covariate columns: ['sample_ID', 'ploidy', 'msStatus', 'library', 'tissue']

✅ Herranz-Ors assets validated


If Nature blocks the automatic Supplementary Table 5 download, download **Supplementary Table 5**
from the open-access Nature article and upload it to Colab, then set `ORG_TABLE5` to that uploaded file.
The notebook will not substitute the DepMap/Achilles reference matrix from the Figshare item.

In [ ]:
# 11) HGNC annotation + AssayFormer vocabulary
HGNC = download(
    "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt",
    SRC / "hgnc_complete_set.txt",
    expected_min_bytes=1_000_000,
)

# Use the released AssayFormer vocabulary directly from Hugging Face.
from huggingface_hub import snapshot_download

CKPT = Path(snapshot_download(
    "Genentech/assayformer",
    revision="4927703e5903984ae0254a9fe8e93ec0b29c27e4"
))
VOCAB = CKPT / "vocab.json"

print("HGNC:", HGNC)
print("Vocab:", VOCAB, VOCAB.exists())

download: hgnc_complete_set.txt
  bytes: 17006236


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

HGNC: /content/external7_sources/hgnc_complete_set.txt
Vocab: /root/.cache/huggingface/hub/models--Genentech--assayformer/snapshots/4927703e5903984ae0254a9fe8e93ec0b29c27e4/vocab.json True


# Build the source screens

These commands use Jeff's existing source-specific builders; no new hit definitions are introduced here.

In [ ]:
# 12) Builder helper
ENV = {**os.environ, "PYTHONPATH": f"{ASSAYLOOP / 'src'}:{ASSAYLOOP / 'scripts/external'}"}

def run_builder(script, args):
    cmd = [str(PY), str(ASSAYLOOP / "scripts/external" / script)] + [str(x) for x in args]
    print("RUN:", " ".join(cmd))
    subprocess.run(cmd, cwd=ASSAYLOOP, env=ENV, check=True)

In [ ]:
# 13) Schukken paired screen — both public interpretations
run_builder("build_schukken_paired.py", [
    "--xlsx", SCHUKKEN,
    "--hgnc", HGNC,
    "--vocab", VOCAB,
    "--out", BUILT / "schukken_paired.pkl",
])

RUN: /content/capstone-jeff/AssayLoop/.venv/bin/python /content/capstone-jeff/AssayLoop/scripts/external/build_schukken_paired.py --xlsx /content/external7_sources/schukken_SD3_beta_scores.xlsx --hgnc /content/external7_sources/hgnc_complete_set.txt --vocab /root/.cache/huggingface/hub/models--Genentech--assayformer/snapshots/4927703e5903984ae0254a9fe8e93ec0b29c27e4/vocab.json --out /content/external7_built/schukken_paired.pkl


In [ ]:
!uv pip install --python /content/capstone-jeff/AssayLoop/.venv/bin/python openpyxl

Using Python 3.11.17 environment at: capstone-jeff/AssayLoop/.venv
Resolved 2 packages in 312ms
Prepared 2 packages in 46ms
Installed 2 packages in 10ms
 + et-xmlfile==2.0.0
 + openpyxl==3.1.5


In [ ]:
import pickle
from pathlib import Path

p = Path("/content/external7_built/schukken_paired.pkl")

print("File exists:", p.exists())
print("File size:", p.stat().st_size if p.exists() else None)

if p.exists():
    with open(p, "rb") as f:
        screens = pickle.load(f)

    print("Screens:", len(screens))

    for s in screens:
        print(
            s.dataset_name,
            "| genes:", s.num_genes,
            "| hits:", s.total_hits
        )

File exists: True
File size: 985579


ModuleNotFoundError: No module named 'assaybench'

In [ ]:
import subprocess, textwrap

code = r"""
import pickle

p = "/content/external7_built/schukken_paired.pkl"

with open(p, "rb") as f:
    screens = pickle.load(f)

print("Screens:", len(screens))

for s in screens:
    print(
        s.dataset_name,
        "| genes:", s.num_genes,
        "| hits:", s.total_hits
    )
"""

result = subprocess.run(
    [
        "/content/capstone-jeff/AssayLoop/.venv/bin/python",
        "-c",
        code
    ],
    text=True,
    capture_output=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

Screens: 2
SCHUKKEN_PAIRED_WGL_paper | genes: 19112 | hits: 188
SCHUKKEN_PAIRED_WGL_rank194 | genes: 19112 | hits: 194



In [ ]:
# 14) Abt
run_builder("build_abt.py", [
    "--summary", ABT,
    "--hgnc", HGNC,
    "--vocab", VOCAB,
    "--out", BUILT / "abt.pkl",
])

RUN: /content/capstone-jeff/AssayLoop/.venv/bin/python /content/capstone-jeff/AssayLoop/scripts/external/build_abt.py --summary /content/external7_sources/abt_gene_summary_FILT.txt --hgnc /content/external7_sources/hgnc_complete_set.txt --vocab /root/.cache/huggingface/hub/models--Genentech--assayformer/snapshots/4927703e5903984ae0254a9fe8e93ec0b29c27e4/vocab.json --out /content/external7_built/abt.pkl


In [ ]:
# 15) Wilson
run_builder("build_wilson.py", [
    "--csv", WILSON,
    "--hgnc", HGNC,
    "--vocab", VOCAB,
    "--out", BUILT / "wilson.pkl",
])

RUN: /content/capstone-jeff/AssayLoop/.venv/bin/python /content/capstone-jeff/AssayLoop/scripts/external/build_wilson.py --csv /content/external7_sources/wilson_ST2_screen_results.csv --hgnc /content/external7_sources/hgnc_complete_set.txt --vocab /root/.cache/huggingface/hub/models--Genentech--assayformer/snapshots/4927703e5903984ae0254a9fe8e93ec0b29c27e4/vocab.json --out /content/external7_built/wilson.pkl


In [ ]:
# 16) Nguyen
run_builder("build_nguyen.py", [
    "--xlsx", NGUYEN,
    "--hgnc", HGNC,
    "--vocab", VOCAB,
    "--out", BUILT / "nguyen.pkl",
])

RUN: /content/capstone-jeff/AssayLoop/.venv/bin/python /content/capstone-jeff/AssayLoop/scripts/external/build_nguyen.py --xlsx /content/external7_sources/nguyen_mmc2_supplemental_tables.xlsx --hgnc /content/external7_sources/hgnc_complete_set.txt --vocab /root/.cache/huggingface/hub/models--Genentech--assayformer/snapshots/4927703e5903984ae0254a9fe8e93ec0b29c27e4/vocab.json --out /content/external7_built/nguyen.pkl


In [ ]:
# 17) Shan
run_builder("build_shan.py", [
    "--xlsx", SHAN,
    "--hgnc", HGNC,
    "--vocab", VOCAB,
    "--out", BUILT / "shan.pkl",
])

RUN: /content/capstone-jeff/AssayLoop/.venv/bin/python /content/capstone-jeff/AssayLoop/scripts/external/build_shan.py --xlsx /content/external7_sources/shan_additional_file2_TableS1.xlsx --hgnc /content/external7_sources/hgnc_complete_set.txt --vocab /root/.cache/huggingface/hub/models--Genentech--assayformer/snapshots/4927703e5903984ae0254a9fe8e93ec0b29c27e4/vocab.json --out /content/external7_built/shan.pkl


In [ ]:
# 18) Herranz-Ors organoids
if Path(ORG_TABLE5).exists() and ORG_COV.exists():
    run_builder("build_organoid.py", [
        "--table5", ORG_TABLE5,
        "--cov", ORG_COV,
        "--hgnc", HGNC,
        "--vocab", VOCAB,
        "--out", BUILT / "organoid.pkl",
    ])
else:
    print("SKIPPED organoid build: Supplementary Table 5 and/or covariates missing.")

RUN: /content/capstone-jeff/AssayLoop/.venv/bin/python /content/capstone-jeff/AssayLoop/scripts/external/build_organoid.py --table5 /content/external7_sources/organoid/organoid_suppl_table5.tsv --cov /content/external7_sources/organoid_data_covariates_ALL-organoids.csv --hgnc /content/external7_sources/hgnc_complete_set.txt --vocab /root/.cache/huggingface/hub/models--Genentech--assayformer/snapshots/4927703e5903984ae0254a9fe8e93ec0b29c27e4/vocab.json --out /content/external7_built/organoid.pkl


In [ ]:
# 19) Samelson
run_builder("build_tau.py", [
    "--all", TAU_ALL,
    "--hits", TAU_HITS,
    "--hgnc", HGNC,
    "--vocab", VOCAB,
    "--out", BUILT / "tau.pkl",
])

RUN: /content/capstone-jeff/AssayLoop/.venv/bin/python /content/capstone-jeff/AssayLoop/scripts/external/build_tau.py --all /content/external7_sources/tau_mmc2_all_genes.xlsx --hits /content/external7_sources/tau_mmc3_hits.xlsx --hgnc /content/external7_sources/hgnc_complete_set.txt --vocab /root/.cache/huggingface/hub/models--Genentech--assayformer/snapshots/4927703e5903984ae0254a9fe8e93ec0b29c27e4/vocab.json --out /content/external7_built/tau.pkl


In [ ]:
import subprocess

code = r"""
import pickle
from pathlib import Path

built = Path("/content/external7_built")

files = [
    "schukken_paired.pkl",
    "abt.pkl",
    "wilson.pkl",
    "nguyen.pkl",
    "shan.pkl",
    "organoid.pkl",
    "tau.pkl",
]

for filename in files:
    path = built / filename

    print("\n" + "=" * 90)
    print(filename)
    print("=" * 90)

    with open(path, "rb") as f:
        screens = pickle.load(f)

    print("screens:", len(screens))

    for s in screens:
        print(
            f"{s.dataset_name:45s}"
            f" genes={s.num_genes:6d}"
            f" hits={s.total_hits:5d}"
        )
"""

result = subprocess.run(
    [
        "/content/capstone-jeff/AssayLoop/.venv/bin/python",
        "-c",
        code,
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)


schukken_paired.pkl
screens: 2
SCHUKKEN_PAIRED_WGL_paper                     genes= 19112 hits=  188
SCHUKKEN_PAIRED_WGL_rank194                   genes= 19112 hits=  194

abt.pkl
screens: 1
ABT_PARCB_dependency                          genes= 14459 hits= 1398

wilson.pkl
screens: 3
WILSON_MERCS_inc                              genes= 18464 hits=  394
WILSON_MERCS_dec                              genes= 18464 hits=  229
WILSON_MERCS_merged                           genes= 18464 hits=  623

nguyen.pkl
screens: 9
NGUYEN_AraC_sensitive                         genes= 19114 hits= 1800
NGUYEN_AraC_resistant                         genes= 19114 hits=  240
NGUYEN_AraC_merged                            genes= 19114 hits= 2040
NGUYEN_Dauno_sensitive                        genes= 19114 hits=  258
NGUYEN_Dauno_resistant                        genes= 19114 hits= 1229
NGUYEN_Dauno_merged                           genes= 19114 hits= 1487
NGUYEN_Etop_sensitive                         genes= 19114 hit

# Verify the exact eleven team targets

Expected counts are taken from Odysseus's `external_sources.json`.

Minor source-gene differences of a few genes are expected for Wilson, Nguyen, and Samelson because
Jeff's public rebuild uses a slightly different duplicate-symbol collapse than the frozen team targets.
The hit count is the critical first check.

Schukken E224 is explicitly ambiguous and is reported separately.

In [ ]:
# 20) Verify and select the team assays
# Unpickle ScreenRecord objects inside Jeff's .venv, then return plain JSON.

import subprocess
import json
import pandas as pd
from pathlib import Path

BUILT = Path("/content/external7_built")

venv_code = r'''
import pickle
import json
from pathlib import Path

BUILT = Path("/content/external7_built")

EXPECTED = {
    "E225_ABT_PARCB_DEPENDENCY": (14459, 1398),
    "E226_WILSON_MERCS_HIGH": (18458, 394),
    "E232_NGUYEN_K562_ARAC_KO_SENSITIZATION": (19111, 240),
    "E236_SHAN_HEPG2_LOVASTATIN": (19063, 368),
    "E236_SHAN_HELA_LOVASTATIN": (19063, 359),
    "E236_SHAN_HELA_LDLR_KO": (19062, 227),
    "E239_ORGANOID_COLO_031": (16562, 984),
    "E239_ORGANOID_OESO_005": (16562, 1683),
    "E239_ORGANOID_OVAR_018": (16562, 1473),
    "E245_SAMELSON_T22_LOWERING": (18818, 728),
}

MAP = {
    "E225_ABT_PARCB_DEPENDENCY":
        ("abt.pkl", "ABT_PARCB_dependency"),

    "E226_WILSON_MERCS_HIGH":
        ("wilson.pkl", "WILSON_MERCS_inc"),

    "E232_NGUYEN_K562_ARAC_KO_SENSITIZATION":
        ("nguyen.pkl", "NGUYEN_AraC_resistant"),

    "E236_SHAN_HEPG2_LOVASTATIN":
        ("shan.pkl", "SHAN_HepG2_lovastatin"),

    "E236_SHAN_HELA_LOVASTATIN":
        ("shan.pkl", "SHAN_HeLa_lovastatin"),

    "E236_SHAN_HELA_LDLR_KO":
        ("shan.pkl", "SHAN_HeLa_LDLRko"),

    "E239_ORGANOID_COLO_031":
        ("organoid.pkl", "ORGANOID_WTSI-COLIVM_031_minLib"),

    "E239_ORGANOID_OESO_005":
        ("organoid.pkl", "ORGANOID_WTSI-OESO_005_minLib"),

    "E239_ORGANOID_OVAR_018":
        ("organoid.pkl", "ORGANOID_WTSI-OVAR_018_minLib"),

    "E245_SAMELSON_T22_LOWERING":
        ("tau.pkl", "TAU_CRISPRi_dec"),
}


def load_screen(file_name, screen_name):
    path = BUILT / file_name

    if not path.exists():
        return None

    with open(path, "rb") as f:
        screens = pickle.load(f)

    by_name = {s.dataset_name: s for s in screens}
    return by_name.get(screen_name)


rows = []
selected10 = []

for team_id, (filename, screen_name) in MAP.items():

    expected_genes, expected_hits = EXPECTED[team_id]

    s = load_screen(filename, screen_name)

    if s is None:
        rows.append({
            "team_id": team_id,
            "screen": screen_name,
            "status": "MISSING BUILD",
            "genes": None,
            "expected_genes": expected_genes,
            "gene_delta": None,
            "hits": None,
            "expected_hits": expected_hits,
            "hit_match": False,
        })
        continue

    selected10.append(s)

    rows.append({
        "team_id": team_id,
        "screen": s.dataset_name,
        "status": "OK" if s.total_hits == expected_hits else "CHECK",
        "genes": s.num_genes,
        "expected_genes": expected_genes,
        "gene_delta": s.num_genes - expected_genes,
        "hits": s.total_hits,
        "expected_hits": expected_hits,
        "hit_match": s.total_hits == expected_hits,
    })


# Save the actual ScreenRecord objects for later experiments.
with open(BUILT / "external10_public_rebuild.pkl", "wb") as f:
    pickle.dump(selected10, f)


# Save plain metadata that normal Colab Python can read.
with open(BUILT / "external10_verification.json", "w") as f:
    json.dump(rows, f, indent=2)

print("Public-rebuild selected screens:", len(selected10))
print("Saved:", BUILT / "external10_public_rebuild.pkl")
print("Saved:", BUILT / "external10_verification.json")
'''

result = subprocess.run(
    [
        "/content/capstone-jeff/AssayLoop/.venv/bin/python",
        "-c",
        venv_code,
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("External subset verification failed.")

# Load only plain JSON into the Colab kernel.
rows = json.loads(
    (BUILT / "external10_verification.json").read_text()
)

verification = pd.DataFrame(rows)

display(verification)

print(
    "\nHit-count matches:",
    int(verification["hit_match"].sum()),
    "/",
    len(verification),
)

Public-rebuild selected screens: 10
Saved: /content/external7_built/external10_public_rebuild.pkl
Saved: /content/external7_built/external10_verification.json



,team_id,screen,status,genes,expected_genes,gene_delta,hits,expected_hits,hit_match
0,E225_ABT_PARCB_DEPENDENCY,ABT_PARCB_dependency,OK,14459,14459,0,1398,1398,True
1,E226_WILSON_MERCS_HIGH,WILSON_MERCS_inc,OK,18464,18458,6,394,394,True
2,E232_NGUYEN_K562_ARAC_KO_SENSITIZATION,NGUYEN_AraC_resistant,OK,19114,19111,3,240,240,True
3,E236_SHAN_HEPG2_LOVASTATIN,SHAN_HepG2_lovastatin,OK,19063,19063,0,368,368,True
4,E236_SHAN_HELA_LOVASTATIN,SHAN_HeLa_lovastatin,OK,19063,19063,0,359,359,True
5,E236_SHAN_HELA_LDLR_KO,SHAN_HeLa_LDLRko,OK,19062,19062,0,227,227,True
6,E239_ORGANOID_COLO_031,ORGANOID_WTSI-COLIVM_031_minLib,OK,16562,16562,0,984,984,True
7,E239_ORGANOID_OESO_005,ORGANOID_WTSI-OESO_005_minLib,OK,16562,16562,0,1683,1683,True
8,E239_ORGANOID_OVAR_018,ORGANOID_WTSI-OVAR_018_minLib,OK,16562,16562,0,1473,1473,True
9,E245_SAMELSON_T22_LOWERING,TAU_CRISPRi_dec,CHECK,18825,18818,7,729,728,False



Hit-count matches: 9 / 10


In [ ]:
# 21) Schukken E224 ambiguity report

import subprocess

code = r'''
import pickle

p = "/content/external7_built/schukken_paired.pkl"

with open(p, "rb") as f:
    screens = pickle.load(f)

for s in screens:
    print(
        f"{s.dataset_name:32s} "
        f"genes={s.num_genes} hits={s.total_hits} "
        f"expected team target: genes=19112 hits=194"
    )

print(
    "\nIMPORTANT: rank194 matches the team's hit COUNT but the team repo "
    "does not expose the frozen 194 hit identities. Do not label this as "
    "an exact reproduction of E224 unless the frozen target/hit list is "
    "obtained from Odysseus."
)
'''

result = subprocess.run(
    [
        "/content/capstone-jeff/AssayLoop/.venv/bin/python",
        "-c",
        code,
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

SCHUKKEN_PAIRED_WGL_paper        genes=19112 hits=188 expected team target: genes=19112 hits=194
SCHUKKEN_PAIRED_WGL_rank194      genes=19112 hits=194 expected team target: genes=19112 hits=194

IMPORTANT: rank194 matches the team's hit COUNT but the team repo does not expose the frozen 194 hit identities. Do not label this as an exact reproduction of E224 unless the frozen target/hit list is obtained from Odysseus.



# What is needed for an exact 11/11 experiment?

If the ten public reconstructions pass their hit-count checks, the remaining exact-data request to Odysseus is very small:

**E224_SCHUKKEN_PAIRED_WGL frozen target / hit identities**

For exact HSA replay, it is also preferable to obtain the frozen external `DESCRIPTION.npy` files,
although those embeddings can be regenerated separately.

The raw public datasets themselves do not need to be requested again.

In [ ]:
# 22) Package verification metadata (not third-party source files)
import shutil, json

report = {
    "verification_rows": verification.to_dict(orient="records") if "verification" in globals() else [],
    "schukken_exact_gap": (
        "Team target has 194 hits; public source supports paper-rule 188 or "
        "rank194 194. Exact frozen hit identities are not in the team repo."
    ),
    "built_dir": str(BUILT),
}

(Path("/content/external7_rebuild_report.json")).write_text(json.dumps(report, indent=2))

print("Report:", "/content/external7_rebuild_report.json")

Report: /content/external7_rebuild_report.json


In [ ]:
import shutil
from pathlib import Path

out = "/content/external7_rebuild_artifacts"

Path(out).mkdir(exist_ok=True)

# Core rebuilt screen files
for f in Path("/content/external7_built").glob("*.pkl"):
    shutil.copy2(f, out)

for f in Path("/content/external7_built").glob("*.json"):
    shutil.copy2(f, out)

# Final reconstruction report
shutil.copy2(
    "/content/external7_rebuild_report.json",
    out
)

zip_path = shutil.make_archive(
    "/content/external7_rebuild_artifacts",
    "zip",
    out
)

print("Created:", zip_path)

from google.colab import files
files.download(zip_path)

Created: /content/external7_rebuild_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import subprocess, shutil, json
from google.colab import _message

REPO_URL = "https://github.com/jhouser220/capstone.git"
BRANCH = "chris-gptoss-handoff"

REPO = Path("/content/capstone-chris-push")
DEST_DIR = (
    REPO
    / "AssayLoop"
    / "experiments"
    / "chris"
    / "external7_rebuild"
)
DEST = DEST_DIR / "external7_data_acquisition.ipynb"

# 1) Clone the existing branch fresh if needed
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    [
        "git", "clone",
        "--branch", BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO),
    ],
    check=True,
)

# 2) Git identity
subprocess.run(
    ["git", "config", "user.name", "Christopher Rizo"],
    cwd=REPO,
    check=True,
)

subprocess.run(
    ["git", "config", "user.email", "christopherrizo@berkeley.edu"],
    cwd=REPO,
    check=True,
)

# 3) Grab the CURRENTLY OPEN COLAB notebook
response = _message.blocking_request(
    "get_ipynb",
    request="",
    timeout_sec=30,
)

if not response or "ipynb" not in response:
    raise RuntimeError("Could not retrieve the current Colab notebook.")

notebook = response["ipynb"]

DEST_DIR.mkdir(parents=True, exist_ok=True)

with open(DEST, "w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=1)

print("Saved current notebook to:")
print(DEST)

# 4) Add README
readme = """# External 7 Public Data Reconstruction

This notebook reconstructs the public versions of the external benchmark
datasets used in the HSA evaluation.

Seven publications are covered:

1. Schukken
2. Abt-Wang
3. Wilson
4. Nguyen
5. Shan
6. Herranz-Ors
7. Samelson

The workflow downloads public source data, normalizes gene identifiers,
builds AssayLoop-compatible screens, and verifies reconstructed gene/hit
counts against the expected benchmark definitions.

This is a public-data reconstruction and is not necessarily identical to
Odysseus's frozen external assets where those assets are unavailable.

Known caveat:
Schukken's public source permits more than one reasonable target
reconstruction. Exact reproduction of the frozen E224 194-hit target
requires the original frozen target identities.
"""

(DEST_DIR / "README.md").write_text(readme)

# 5) Secret scan
danger_patterns = ["gsk_", "gho_", "sk-proj-"]

for p in DEST_DIR.rglob("*"):
    if not p.is_file():
        continue
    txt = p.read_text(errors="ignore")
    for pattern in danger_patterns:
        if pattern in txt:
            raise RuntimeError(
                f"Potential secret pattern '{pattern}' found in {p}"
            )

print("Secret scan passed.")

# 6) Stage + commit
subprocess.run(
    ["git", "add", "AssayLoop/experiments/chris/external7_rebuild"],
    cwd=REPO,
    check=True,
)

print("\n===== STATUS =====")
print(
    subprocess.check_output(
        ["git", "status", "--short"],
        cwd=REPO,
        text=True,
    )
)

commit = subprocess.run(
    [
        "git", "commit",
        "-m", "Add external benchmark reconstruction notebook",
    ],
    cwd=REPO,
    text=True,
    capture_output=True,
)

print("\nCOMMIT RETURN CODE:", commit.returncode)
print(commit.stdout)
print(commit.stderr)

# 7) Push
if commit.returncode == 0:
    push = subprocess.run(
        ["git", "push", "origin", BRANCH],
        cwd=REPO,
        text=True,
        capture_output=True,
    )

    print("\nPUSH RETURN CODE:", push.returncode)
    print(push.stdout)
    print(push.stderr)